<a href="https://colab.research.google.com/github/VitorBZS/PLN-A940-M-D.S.M.-297-20262/blob/main/Aula_05_Pr%C3%A1tica_Passo_a_Passo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q nltk spacy pandas
!python -m spacy download pt_core_news_md -q

import nltk
import spacy
import pandas as pd

nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)

nlp = spacy.load("pt_core_news_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 9.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


#1. Prática leve: explorando similiariedade

In [8]:
pares = [
    ("apartamento", "imóvel"),
    ("corretor", "vendedor"),
    ("cachorro", "gato"),
]

for palavra_a, palavra_b in pares:
  a = nlp(palavra_a)
  b = nlp(palavra_b)
  print(f"Similiariedade {palavra_a} x {palavra_b}: {a.similarity(b):.3f}")


Similiariedade apartamento x imóvel: 0.667
Similiariedade corretor x vendedor: 0.624
Similiariedade cachorro x gato: 0.837


#2. Exercício guiado: revisando tudo, passo a passo

**Frase**: "O corretor negociou a venda do apartamento com o cliente."

In [10]:
frase_guiada = "O corretor negociou a venda do apartamento com o cliente."
doc_guiado = nlp(frase_guiada)

print("Frase:", frase_guiada)

Frase: O corretor negociou a venda do apartamento com o cliente.


##Passo 1 - Tokenização

In [12]:
tokens = [token.text for token in doc_guiado]
print("Tokens:", tokens)

Tokens: ['O', 'corretor', 'negociou', 'a', 'venda', 'do', 'apartamento', 'com', 'o', 'cliente', '.']


##Passo 2 - Remoção de pontuação e stopwords

In [13]:
tokens_limpos = [token.text for token in doc_guiado if not token.is_punct and not token.is_stop]
print("Tokens limpos:", tokens_limpos)

Tokens limpos: ['corretor', 'negociou', 'venda', 'apartamento', 'cliente']


##Passo 3 - Steamming e Lematização

In [14]:
from nltk.stem import RSLPStemmer

stemmer = RSLPStemmer()
stems = [stemmer.stem(t) for t in tokens_limpos]

doc_limpo = nlp(" ".join(tokens_limpos))
lemas = [token.lemma_ for token in doc_limpo]

comparacao = pd.DataFrame({
    "Token": tokens_limpos,
    "Stemming (RSLP)": stems,
    "Lematização (spaCy)": lemas
})
comparacao


,Token,Stemming (RSLP),Lematização (spaCy)
0,corretor,corre,corretor
1,negociou,negoci,negociar
2,venda,vend,venda
3,apartamento,apart,apartamento
4,cliente,client,cliente


##Passo 4 - POS Tagging

In [16]:
dados_pos = [{"Token": t.text, "POS": t.pos_} for t in doc_guiado]
pd.DataFrame(dados_pos)

,Token,POS
0,O,DET
1,corretor,NOUN
2,negociou,VERB
3,a,DET
4,venda,NOUN
5,do,ADP
6,apartamento,NOUN
7,com,ADP
8,o,DET
9,cliente,NOUN


##Passo 5 - Sintagma Nominal e Sintagma Verbal

In [18]:
raiz = [token for token in doc_guiado if token.dep_ == "ROOT"][0]
sujeito = [token for token in doc_guiado if token.dep_ == "nsubj"][0]

sintagma_nominal = list(sujeito.subtree)
sintagma_verbal = [t for t in raiz.subtree if t not in sintagma_nominal]

print("Sintagma Nominal(SN):", [t.text for t in sintagma_nominal])
print("Sintagma Nominal(SN):", [t.text for t in sintagma_verbal])

Sintagma Nominal(SN): ['O', 'corretor']
Sintagma Nominal(SN): ['negociou', 'a', 'venda', 'do', 'apartamento', 'com', 'o', 'cliente', '.']


##Passo 6 - Similiariedade

In [17]:
palavra_a = nlp("corretor")
palavra_b = nlp("venda")

print(f"Similaridade corretor x venda: {palavra_a.similarity(palavra_b):.3f}")

Similaridade corretor x venda: 0.367
